<!-- Add this markdown cell at the very beginning -->
# Fine-Tuning Tutorial: Sentiment Classification with DistilBERT

This notebook demonstrates how to fine-tune a pre-trained transformer model (DistilBERT) for sentiment classification using the Hugging Face ecosystem. We'll walk through the entire process from loading data to making predictions with our fine-tuned model.

## What We'll Cover:
1. **Dataset Loading** - Load and explore sentiment analysis datasets
2. **Tokenization** - Prepare text data for the model
3. **Model Setup** - Load a pre-trained model for classification
4. **Training Configuration** - Set up training parameters
5. **Fine-tuning** - Train the model on our dataset
6. **Inference** - Use the trained model for predictions

---

## 1. Loading Required Libraries and Datasets

First, we'll import the necessary libraries and load our datasets. We'll use two popular sentiment analysis datasets to demonstrate different options.

In [4]:
from datasets import load_dataset

### Loading Datasets

We'll load two different sentiment analysis datasets:
- **Amazon Polarity**: Amazon product reviews with binary sentiment (positive/negative)
- **GLUE SST-2**: Stanford Sentiment Treebank with binary sentiment classification

This shows you different dataset options available through Hugging Face datasets library.

In [5]:
raw_datasets_1=load_dataset("amazon_polarity")
raw_datasets_2=load_dataset("glue", "sst2")

In [6]:
raw_datasets_2

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [20]:
raw_datasets_2["train"].features

{'sentence': Value(dtype='string', id=None),
 'label': ClassLabel(names=['negative', 'positive'], id=None),
 'idx': Value(dtype='int32', id=None)}

### Exploring the Dataset Structure

Let's examine the structure of our chosen dataset (GLUE SST-2) to understand what data we're working with.

The dataset has:
- **sentence**: The text data (input)
- **label**: The sentiment label (0 = negative, 1 = positive)
- **idx**: Index identifier for each sample

Let's examine the label distribution:

In [ ]:
raw_datasets_2["train"]["label"]

AttributeError: 'list' object has no attribute 'isunique'

## 2. Tokenization Setup

Before we can train our model, we need to convert text into tokens that the model can understand. We'll use DistilBERT's tokenizer, which is compatible with BERT but more efficient.

### Loading the Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Creating a Tokenization Function

We need to create a function that will tokenize our text data. The function should:
- Take the text from the "sentence" field
- Apply truncation to handle long sequences
- Return tokenized inputs (input_ids, attention_mask, etc.)

In [10]:
def tokenizer_fun(data):
    return tokenizer(data["sentence"], truncation=True)

In [11]:
tokenized_dataset=raw_datasets_2.map(tokenizer_fun,batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [24]:
tokenized_dataset["train"]

Dataset({
    features: ['sentence', 'label', 'idx', 'input_ids', 'attention_mask'],
    num_rows: 67349
})

Now our dataset includes:
- **input_ids**: Token IDs that represent our text
- **attention_mask**: Mask indicating which tokens are actual content vs padding
- **label**: Original sentiment labels (preserved)

---

## 3. Training Configuration

Next, we'll set up the training parameters. These control how the model will be trained.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,
)


### Training Arguments Explained:
- **output_dir**: Where to save model checkpoints and results
- **eval_strategy**: When to run evaluation (here: after each epoch)
- **save_strategy**: When to save model checkpoints
- **num_train_epochs**: How many times to go through the entire dataset

---

## 4. Model Setup

We'll load a pre-trained DistilBERT model and adapt it for sequence classification (sentiment analysis).

In [27]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Model Configuration:
- **Base Model**: DistilBERT (faster, smaller version of BERT)
- **Task**: Sequence Classification
- **num_labels=2**: Binary classification (positive/negative sentiment)

The model automatically adds a classification head on top of the base DistilBERT model.

---

## 5. Evaluation Metrics

We'll define how to measure our model's performance during training.

In [28]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pred=predictions.argmax(axis=1)
    acc=accuracy_score(labels, pred)
    return {"accuracy": acc}

### Metrics Function Explained:
- **predictions**: Raw model outputs (logits/probabilities)
- **argmax(axis=1)**: Converts probabilities to predicted class labels
- **accuracy_score**: Compares predicted labels with true labels
- Returns a dictionary with metric name and value

---

## 6. Setting Up the Trainer

The Trainer class handles the training loop, evaluation, and other training logistics.

In [ ]:


from transformers import Trainer, AutoModelForSequenceClassification

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

### Trainer Components:
- **model**: The model to train
- **args**: Training configuration we defined earlier
- **train_dataset**: Tokenized training data
- **eval_dataset**: Tokenized validation data for monitoring performance
- **tokenizer**: Needed for proper data collation
- **compute_metrics**: Function to calculate performance metrics

---

## 7. Fine-Tuning the Model

Now we'll start the actual training process. This will fine-tune the pre-trained DistilBERT on our sentiment analysis task.

In [ ]:
trainer.train()

### What Happens During Training:
1. **Forward Pass**: Model processes batches of tokenized text
2. **Loss Calculation**: Compares predictions with true labels
3. **Backward Pass**: Calculates gradients
4. **Parameter Update**: Adjusts model weights
5. **Evaluation**: Runs on validation set after each epoch
6. **Checkpointing**: Saves model state for recovery

The training will show progress with loss and evaluation metrics.

---

## 8. Using the Fine-Tuned Model

After training, we can use our model for making predictions on new text.

### Loading the Model for Inference

We'll create a pipeline that makes it easy to classify new text:

In [31]:
from transformers import pipeline

classifier=pipeline("text-classification", model="mymodel", tokenizer=tokenizer)

### Pipeline Explanation:
- **"text-classification"**: Specifies the task type
- **model="mymodel"**: Path to our saved fine-tuned model
- **tokenizer**: The same tokenizer used during training

### Making Predictions

Now we can classify new text examples:

In [33]:
classifier("This is a bad movie")  # Example usage of the classifier

[{'label': 'LABEL_0', 'score': 0.9982823133468628}]

### Understanding the Output:
The classifier returns:
- **LABEL_0**: Negative sentiment (class 0)
- **LABEL_1**: Positive sentiment (class 1)
- **score**: Confidence score (probability) for the prediction

---

## Summary

🎉 **Congratulations!** You've successfully:

1. ✅ **Loaded** a sentiment analysis dataset
2. ✅ **Tokenized** text data for model input
3. ✅ **Configured** training parameters
4. ✅ **Fine-tuned** a pre-trained DistilBERT model
5. ✅ **Evaluated** model performance
6. ✅ **Created** a classifier for new predictions

### Key Takeaways:
- **Pre-trained models** provide a strong starting point
- **Fine-tuning** adapts models to specific tasks
- **Tokenization** is crucial for text preprocessing
- **Evaluation metrics** help monitor training progress
- **Pipelines** make inference simple and intuitive

### Next Steps:
- Try different datasets or model architectures
- Experiment with training parameters
- Add more sophisticated evaluation metrics
- Deploy your model for real-world use

---